# 04 · CUB — data analysis (raw annotations) & the label-standardization critique

**No processed pkls or model needed** — reads the raw CUB-200-2011 image-level
attribute labels directly. Two questions, both load-bearing for the CUB story:

1. Do CUB attributes **vary within a species** at the image level? → if yes, the
   matched-pair **recall gap is powered** on CUB (FunnyBirds was 0% — see nb 01).
2. What does the CBM **majority-vote-to-class-level** standardization (Koh et al.)
   *do* to that variation? → it flattens it to 0, and relabels a measurable share of
   images to the species-typical value. **That relabeling is where backwash is
   trained in** (§3b).

Analysis is on the **112 CBM-trained attributes** (rare attrs excluded → noise-robust).

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO = Path.cwd().parent
try:
    sys.path.insert(0, str(REPO/"analysis")); from plotting import set_paper_style; set_paper_style()
except Exception: pass
plt.rcParams["figure.dpi"]=120
# locate raw CUB
cands=[REPO.parent/"data"/"CUB_200_2011", Path(os.environ.get("CURATED_DATA",""))/"CUB_200_2011",
       Path(os.environ.get("CURATED_DATA","")).parent/"data"/"CUB_200_2011"]
CUB=next((p for p in cands if (p/"image_class_labels.txt").exists()), None)
assert CUB is not None, f"CUB_200_2011 not found in {cands}"
print("CUB:", CUB)

def load_raw(root):
    imgs,attrs,pres=[],[],[]
    with open(root/"attributes"/"image_attribute_labels.txt") as f:
        for ln in f:
            p=ln.split()
            if len(p)>=3: imgs.append(int(p[0])); attrs.append(int(p[1])); pres.append(int(p[2]))
    imgs=np.array(imgs); attrs=np.array(attrs); pres=np.array(pres,dtype=np.int8)
    M=np.zeros((imgs.max(),attrs.max()),np.int8); M[imgs-1,attrs-1]=pres
    cls=np.loadtxt(root/"image_class_labels.txt",dtype=np.int64)
    y=np.zeros(M.shape[0],np.int64); y[cls[:,0]-1]=cls[:,1]
    return M,y
M,y=load_raw(CUB)
try:
    sys.path.insert(0,str(REPO/"external"/"minimal_cbm")); from src.datasets.cub200 import USED_ATTRIBUTES
    used=np.array(sorted(USED_ATTRIBUTES))-1; print(f"restricting to {len(used)} CBM-used attributes")
except Exception as e:
    used=np.arange(M.shape[1]); print("USED_ATTRIBUTES unavailable -> all", len(used), "attrs:", e)
Mu=M[:,used]
print(f"{Mu.shape[0]} images · {len(np.unique(y))} species · {Mu.shape[1]} attributes")

## 1 · Within-species variation — is the recall gap powered on CUB?
Within-species std of each attribute on the raw image-level labels. Anything >0 means
the attribute is *not* constant across a species' images → matched-pair recall gap has
signal. Compare full CUB vs CUB70 (first 70 classes).

In [ ]:
def frac_vary(Mx, yx):
    w=np.array([Mx[yx==c].std(0) for c in np.unique(yx)])
    return 1-float(np.mean(w==0)), w
keep = y<=70
fv_full,w_full = frac_vary(Mu, y)
fv_70,_        = frac_vary(Mu[keep], y[keep])
print(f"full CUB: {fv_full*100:.1f}% of (species,attr) pairs vary within a species")
print(f"CUB70   : {fv_70*100:.1f}%   (FunnyBirds was 0% -> recall gap is a CUB tool)")
fig,ax=plt.subplots(1,2,figsize=(11,3.4))
ax[0].bar(["full CUB","CUB70","FunnyBirds"],[fv_full*100,fv_70*100,0],color=["#0072B2","#5B8C5A","#bbbbbb"])
ax[0].set_ylabel("% (species,attr) pairs varying"); ax[0].set_title("Within-species variation (recall-gap power)")
ax[1].hist(w_full[w_full>0].ravel(),bins=40,color="#0072B2"); ax[1].set_xlabel("within-species std (nonzero pairs)")
ax[1].set_ylabel("count"); ax[1].set_title("Spread of within-species variation (full CUB)"); plt.tight_layout()

## 2 · The CBM standardization — majority-vote erases the variation
Koh et al. majority-vote each attribute to the species level ("all birds of a species
share concept annotations"). By construction that makes within-species variation **0**.
We count the image-attribute cells that get **relabeled** to the species-typical value —
those images are trained to report a concept that contradicts their own annotation.

In [ ]:
def relabel_stats(Mx, yx):
    classes=np.unique(yx); persp=np.array([Mx[yx==c].mean(0) for c in classes])
    mv=(persp>=0.5).astype(np.int8); flips=total=0
    for i,c in enumerate(classes):
        b=Mx[yx==c]; flips+=int((b!=mv[i]).sum()); total+=b.size
    return flips, total, persp
f_full,t_full,persp_full = relabel_stats(Mu, y)
f_70,t_70,_ = relabel_stats(Mu[keep], y[keep])
print(f"full CUB: {f_full}/{t_full} cells relabeled by majority-vote = {100*f_full/t_full:.1f}%")
print(f"CUB70   : {f_70}/{t_70} = {100*f_70/t_70:.1f}%")
fig,ax=plt.subplots(figsize=(6,3.3))
ax.bar(["raw\n(varies)","after CBM\nmajority-vote"],[fv_full*100,0],color=["#0072B2","#D55E00"])
ax.set_ylabel("% pairs varying within species"); ax.set_title(f"Standardization erases variation\n({100*f_full/t_full:.1f}% of labels overwritten)")
for i,v in enumerate([fv_full*100,0]): ax.text(i,v+1,f"{v:.1f}%",ha="center")

## 3 · Genuinely ambiguous attributes — the "belly white or yellow?" cases
A single flipped label could be annotator noise; an attribute present in **30–70%** of a
species' images is *real* within-species variation (lighting / viewpoint / polymorphism)
that majority-voting still flattens. These are the hardest cases to dismiss as noise.

In [ ]:
amb = (persp_full>=0.3)&(persp_full<=0.7)
print(f"ambiguous (species,attr) pairs (present 30-70%): {100*amb.mean():.1f}% of all pairs")
fig,ax=plt.subplots(figsize=(6.5,3.3))
ax.hist(persp_full.ravel(),bins=50,color="#0072B2")
ax.axvspan(0.3,0.7,color="#D55E00",alpha=0.2,label="ambiguous band (30–70%)")
ax.set_xlabel("P(attribute present) within a species"); ax.set_ylabel("count of (species,attr) pairs")
ax.set_title("Per-species attribute prevalence — mass in the ambiguous band is real variation"); ax.legend()

## Takeaway (§3b, measured)
- CUB has **real within-species attribute variation** (≈60% of pairs) — so the recall
  gap is a genuine test here, unlike FunnyBirds.
- The CBM **majority-vote standardization erases all of it** and **overwrites ≈8–9% of
  image-attribute labels** to the species-typical value.
- ≈10% of species-attribute pairs are genuinely split (30–70%) — not noise.

So on the very dataset CBM/MCBM are validated on, the concept labels are made
`concept = f(species)` **by preprocessing**, and a measurable fraction of images are
trained against their own evidence. That is the mechanism the model-side backwash
(next: trained CUB / CUB70) then exhibits. *Numbers provisional (single pass); see
`RESULTS.md`.*